# Reflection Generation Pipeline

This notebook connects:
- Artwork context
- Prompt design
- AI provider abstraction

It demonstrates the full reflection generation workflow, including:
- Initial reflection generation
- User feedback incorporation
- Revised reflection generation

This notebook simulates the AI logic used by the backend service.


In [1]:
from dataclasses import dataclass
from typing import Optional
from abc import ABC, abstractmethod


In [2]:
@dataclass
class ArtworkContext:
    title: str
    artist: Optional[str]
    description: str


In [3]:
class AIProvider(ABC):

    @abstractmethod
    def generate_reflection(self, prompt: str) -> str:
        pass


In [4]:
class MockAIProvider(AIProvider):

    def generate_reflection(self, prompt: str) -> str:
        if "Rewrite the reflection" in prompt:
            return (
                "The artwork feels gently inviting, creating a warmer sense of calm "
                "that allows personal emotions to settle naturally and without pressure."
            )

        return (
            "The artwork offers a quiet presence, leaving room for personal feelings "
            "to emerge slowly and without needing to be clearly defined."
        )


In [5]:
SYSTEM_PROMPT = """
You are a reflective writing assistant helping a user express personal
feelings about an artwork.

Rules:
- Do NOT analyze or explain
- Do NOT teach or interpret
- Avoid certainty and authority
- Write a single short paragraph
- Do NOT ask questions
""".strip()


In [6]:
def build_initial_prompt(context: ArtworkContext) -> str:
    artist_text = f"by {context.artist}" if context.artist else "by an unknown artist"

    return f"""
SYSTEM:
{SYSTEM_PROMPT}

USER:
The user is viewing an artwork titled "{context.title}" {artist_text}.

Artwork description:
{context.description}

Write a reflective paragraph to help the user engage emotionally.
""".strip()


In [7]:
def build_revision_prompt(
    context: ArtworkContext,
    previous_reflection: str,
    user_feedback: str
) -> str:
    return f"""
SYSTEM:
{SYSTEM_PROMPT}

USER:
Artwork context:
Title: {context.title}
Description: {context.description}

Previous reflection:
{previous_reflection}

User feedback:
{user_feedback}

Rewrite the reflection incorporating the user's feedback.
""".strip()


In [8]:
def generate_reflection_pipeline(
    provider: AIProvider,
    context: ArtworkContext,
    previous_reflection: Optional[str] = None,
    user_feedback: Optional[str] = None
) -> str:

    if previous_reflection and user_feedback:
        prompt = build_revision_prompt(
            context,
            previous_reflection,
            user_feedback
        )
    else:
        prompt = build_initial_prompt(context)

    return provider.generate_reflection(prompt)


In [9]:
provider = MockAIProvider()

artwork = ArtworkContext(
    title="Evening Stillness",
    artist=None,
    description="Soft hills under a fading sky, with muted colors and gentle light."
)

initial_reflection = generate_reflection_pipeline(
    provider=provider,
    context=artwork
)

print("Initial Reflection:\n")
print(initial_reflection)


Initial Reflection:

The artwork offers a quiet presence, leaving room for personal feelings to emerge slowly and without needing to be clearly defined.


In [10]:
user_feedback = (
    "This feels a bit distant. Make it warmer and more personal."
)


In [11]:
revised_reflection = generate_reflection_pipeline(
    provider=provider,
    context=artwork,
    previous_reflection=initial_reflection,
    user_feedback=user_feedback
)

print("Revised Reflection:\n")
print(revised_reflection)


Revised Reflection:

The artwork feels gently inviting, creating a warmer sense of calm that allows personal emotions to settle naturally and without pressure.


In [12]:
user_confirms = True  # simulated user confirmation

if user_confirms:
    final_reflection = revised_reflection
    print("Final reflection approved and ready to be saved.")
else:
    print("Reflection requires further revision.")


Final reflection approved and ready to be saved.


## Engineering Notes

- This pipeline does not depend on any specific AI provider
- User feedback is treated as content, not instructions
- Reflection persistence requires explicit user confirmation
- No ML training or fine-tuning is involved
- This logic maps directly to backend service behavior
